## Convert to ONNX
ONNX is another way to convert large TF-Models to mobile devices. It is more broad then tflite and enables converstion of pytorch models.

In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from PIL import Image

In [17]:
import onnx
import tf2onnx
import onnxruntime as ort

In [3]:
print(f"✓ NumPy: {np.__version__}")
print(f"✓ TensorFlow: {tf.__version__}")
print(f"✓ ONNX: {onnx.__version__}")
print(f"✓ tf2onnx: {tf2onnx.__version__}")

✓ NumPy: 1.26.4
✓ TensorFlow: 2.16.1
✓ ONNX: 1.17.0
✓ tf2onnx: 1.16.1


In [9]:
# Create a new model with fixed input shape
ham_model = load_model("models/ham10000_poc.keras")
inputs = tf.keras.Input(shape=(150, 200, 3))
outputs = ham_model(inputs)
fixed_model = tf.keras.Model(inputs=inputs, outputs=outputs)

## Acutual conversion

In [13]:
spec = (tf.TensorSpec((None, 150, 200, 3), tf.float32, name="input"),)
onnx_model, _ = tf2onnx.convert.from_keras(fixed_model, input_signature=spec, opset=13)
onnx.save(onnx_model, "models/ham10000.onnx")

## Test original keras model

In [15]:
cancer_img = image.load_img("cancer_mole.jpg")
cancer_img = cancer_img.convert("RGB")
img_array = image.img_to_array(cancer_img)
img_array = np.expand_dims(img_array, axis=0)

img_array = img_array / 255.0

prediction = ham_model.predict(img_array)

percentages = prediction * 100

probs = np.argmax(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step


In [16]:
for i, percent in enumerate(percentages[0]):
    print(f"Class {i}: {percent:.2f}%")

Class 0: 0.99%
Class 1: 1.49%
Class 2: 14.02%
Class 3: 0.04%
Class 4: 49.46%
Class 5: 33.91%
Class 6: 0.11%


## Test ONNX Model

In [21]:
# Load ONNX model
onnx_session = ort.InferenceSession("models/ham10000.onnx")

# Get input/output details
input_name = onnx_session.get_inputs()[0].name
output_name = onnx_session.get_outputs()[0].name
input_shape = onnx_session.get_inputs()[0].shape

print(f"✓ Input name: {input_name}")
print(f"✓ Input shape: {input_shape}")
print(f"✓ Output name: {output_name}")

# Load and preprocess image
cancer_img = Image.open("cancer_mole.jpg")
cancer_img = cancer_img.convert("RGB")
cancer_img = cancer_img.resize((200, 150), Image.BILINEAR)  # Resize to match model input

img_array = np.array(cancer_img, dtype=np.float32)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

print(f"\nTest image shape: {img_array.shape}")
print(f"Image value range: {img_array.min():.3f} - {img_array.max():.3f}")

# Run inference
outputs = onnx_session.run([output_name], {input_name: img_array})
predictions = outputs[0]

print(f"\n✓ ONNX predictions: {predictions}")
print(f"✓ Predicted class: {np.argmax(predictions[0])}")
print(f"\n✓ Class probabilities:")
for i, prob in enumerate(predictions[0]):
    print(f"   Class {i}: {prob:.6f} {'← PREDICTED' if i == np.argmax(predictions[0]) else ''}")

✓ Input name: input
✓ Input shape: ['unk__145', 150, 200, 3]
✓ Output name: sequential

Test image shape: (1, 150, 200, 3)
Image value range: 0.012 - 0.976

✓ ONNX predictions: [[5.5676278e-06 8.1757889e-07 6.7765848e-03 8.6010479e-09 1.0673969e-01
  8.8647723e-01 7.3366074e-08]]
✓ Predicted class: 5

✓ Class probabilities:
   Class 0: 0.000006 
   Class 1: 0.000001 
   Class 2: 0.006777 
   Class 3: 0.000000 
   Class 4: 0.106740 
   Class 5: 0.886477 ← PREDICTED
   Class 6: 0.000000 


In [22]:
ham_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resize (Resizing)               │ (None, 150, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 150, 200, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 150, 200, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 150, 200, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 150, 200, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 150, 200, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 100, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 75, 100, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 75, 100, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 37, 50, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 18, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,340,176 (58.52 MB)

 Trainable params: 7,670,087 (29.26 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,670,089 (29.26 MB)